# Kiltrazo: medir el acierto del reconocimiento

Este cuaderno mide qué tan bien reconoce Kiltrazo a un perro, con **los mismos modelos y el mismo proceso que la app**:

1. el detector de cabezas propio (`public/models/pet-head.onnx`) recorta la cara, con el mismo margen que el escáner;
2. DINOv2-small (el mismo archivo ONNX que descarga el celular) saca la huella de 384 números, sumando la foto y su espejo;
3. se compara contra **todos** los perros registrados, igual que `report_found`: gana el más parecido y, si pasa el umbral (0.78), se avisa a su dueño.

Usa el set público **DogFaceNet** (unos 1.400 perros con varias fotos de cada uno). Solo se usa para medir, no para entrenar. Si se publica el resultado, hay que citar: *Mougeot, Li y Jia, "A Deep Learning Approach for Dog Face Verification and Recognition", PRICAI 2019*.

**Cómo usarlo:** *Entorno de ejecución → Ejecutar todo*. No necesita GPU. Tarda unos 20 a 40 minutos. Al final muestra el resumen, un gráfico y descarga `resultados_reconocimiento.json` y `grafico_reconocimiento.png`.

In [ ]:
# 1. Preparar
import os, sys, json, glob, random, zipfile, urllib.request, io
if 'google.colab' in sys.modules:
    !pip -q install onnxruntime huggingface_hub
import numpy as np
from PIL import Image, ImageOps
import onnxruntime as ort

UMBRAL = 0.78        # el de la app (src/biometrics.js)
SUGERENCIA = 0.63    # desde aquí la app muestra "¿es esta?"
MAX_PERROS = None    # por ejemplo 200 para una prueba rápida
CARPETA = os.environ.get('KILTRAZO_DATOS', '/content/dogfacenet')
SEMILLA = 7
REPO = 'https://raw.githubusercontent.com/andresmaturana-ui/petsafe/main'

## 2. Descargar las fotos

In [ ]:
EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

def fotos_por_perro(raiz):
    # Cada carpeta con fotos es un perro. Se usa solo el nombre de la carpeta,
    # por si el mismo perro aparece en dos partes del set (por ejemplo train y test).
    perros = {}
    for ruta in glob.glob(os.path.join(raiz, '**', '*'), recursive=True):
        if ruta.lower().endswith(EXT):
            perros.setdefault(os.path.basename(os.path.dirname(ruta)), []).append(ruta)
    return {k: sorted(v) for k, v in perros.items() if len(v) >= 2}

def descargar(raiz):
    os.makedirs(raiz, exist_ok=True)
    if fotos_por_perro(raiz):
        return
    try:
        # Zenodo: set completo publicado por los autores.
        reg = json.load(urllib.request.urlopen('https://zenodo.org/api/records/12578449', timeout=60))
        zips = [f for f in reg['files'] if f['key'].lower().endswith('.zip')]
        print('Archivos en Zenodo:', [(f['key'], f['size'] // 2**20) for f in zips], 'MB')
        # Preferir la versión de 224 px (más liviana); si no, el zip más chico.
        f = sorted(zips, key=lambda f: ('224' not in f['key'], f['size']))[0]
        destino = os.path.join(raiz, f['key'])
        print('Descargando', f['key'], '...')
        urllib.request.urlretrieve(f['links']['self'], destino)
        with zipfile.ZipFile(destino) as z:
            z.extractall(raiz)
        os.remove(destino)
    except Exception as e:
        print('Zenodo no respondió (', e, '), uso la copia de Hugging Face')
        from huggingface_hub import snapshot_download
        import pandas as pd
        d = snapshot_download('dimidagd/DogFaceNet_224resize', repo_type='dataset')
        n = 0
        for pq in glob.glob(os.path.join(d, '**', '*.parquet'), recursive=True):
            df = pd.read_parquet(pq)
            col_img = next(c for c in df.columns if isinstance(df[c].iloc[0], dict) and 'bytes' in df[c].iloc[0])
            col_id = next(c for c in df.columns if c != col_img)
            for _, fila in df.iterrows():
                carpeta = os.path.join(raiz, str(fila[col_id]))
                os.makedirs(carpeta, exist_ok=True)
                Image.open(io.BytesIO(fila[col_img]['bytes'])).convert('RGB').save(os.path.join(carpeta, f'{n}.jpg'))
                n += 1

descargar(CARPETA)
perros = fotos_por_perro(CARPETA)
claves = sorted(perros)
random.Random(SEMILLA).shuffle(claves)
if MAX_PERROS:
    claves = claves[:MAX_PERROS]
perros = {k: perros[k] for k in claves}
print(f'{len(perros)} perros, {sum(map(len, perros.values()))} fotos')
assert len(perros) >= 20, 'No encontré carpetas con fotos de cada perro: revisa qué se descargó en ' + CARPETA

## 3. Cargar los modelos de la app

In [ ]:
def bajar(url, nombre):
    if not os.path.exists(nombre):
        urllib.request.urlretrieve(url, nombre)
    return nombre

opciones = ort.SessionOptions()
cabeza = ort.InferenceSession(os.environ.get('KILTRAZO_DETECTOR') or bajar(REPO + '/public/models/pet-head.onnx', 'pet-head.onnx'), opciones)
if os.environ.get('KILTRAZO_DINO'):
    dino_onnx = os.path.join(os.environ['KILTRAZO_DINO'], 'onnx/model_quantized.onnx')
    pre = json.load(open(os.path.join(os.environ['KILTRAZO_DINO'], 'preprocessor_config.json')))
else:
    from huggingface_hub import hf_hub_download
    dino_onnx = hf_hub_download('Xenova/dinov2-small', 'onnx/model_quantized.onnx')
    pre = json.load(open(hf_hub_download('Xenova/dinov2-small', 'preprocessor_config.json')))
dino = ort.InferenceSession(dino_onnx, opciones)
print('Modelos listos')

## 4. Sacar la huella de cada foto
Mismo proceso que `src/scanner.js` y `server/app.py`.

In [ ]:
TAM = 224

def buscar_cabeza(img):
    # Igual que locateHead(): se achica sin deformar, relleno gris, salida YOLO [1, 5, N].
    n = 320
    k = n / max(img.width, img.height)
    w, h = round(img.width * k), round(img.height * k)
    dx, dy = (n - w) / 2, (n - h) / 2
    caja = Image.new('RGB', (n, n), (114, 114, 114))
    caja.paste(img.resize((w, h), Image.BILINEAR), (round(dx), round(dy)))
    x = (np.asarray(caja, np.float32) / 255).transpose(2, 0, 1)[None]
    out = cabeza.run(None, {cabeza.get_inputs()[0].name: x})[0][0]
    i = int(out[4].argmax())
    if out[4, i] < 0.4:
        return None
    cx, cy, bw, bh = (float(v) for v in out[:4, i])
    return ((cx - dx) / k, (cy - dy) / k, bw / k, bh / k)

def recorte(img, caja):
    # Igual que cropAround(): cuadrado 1.25 veces la cabeza, relleno gris.
    if caja:
        cx, cy, w, h = caja
        lado = min(max(w, h) * 1.25, max(img.width, img.height))
    else:
        cx, cy, lado = img.width / 2, img.height / 2, min(img.width, img.height)
    sx, sy = cx - lado / 2, cy - lado / 2
    k = TAM / lado
    lienzo = Image.new('RGB', (TAM, TAM), (128, 128, 128))
    grande = img.resize((max(1, round(img.width * k)), max(1, round(img.height * k))), Image.LANCZOS)
    lienzo.paste(grande, (round(-sx * k), round(-sy * k)))
    return lienzo

def cls(img):
    s = pre['size']['shortest_edge']
    k = s / min(img.width, img.height)
    img = img.resize((round(img.width * k), round(img.height * k)), Image.BICUBIC)
    c = pre['crop_size']['height']
    izq, arr = (img.width - c) // 2, (img.height - c) // 2
    img = img.crop((izq, arr, izq + c, arr + c))
    x = np.asarray(img, np.float32) * pre['rescale_factor']
    x = (x - np.array(pre['image_mean'], np.float32)) / np.array(pre['image_std'], np.float32)
    return dino.run(None, {dino.get_inputs()[0].name: x.transpose(2, 0, 1)[None]})[0][0, 0]

def huella(ruta):
    img = Image.open(ruta).convert('RGB')
    caja = buscar_cabeza(img)
    cara = recorte(img, caja)
    v = cls(cara) + cls(ImageOps.mirror(cara))
    return v / (np.linalg.norm(v) or 1), caja is not None

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **k: x

H, detectadas = {}, 0
for perro, rutas in tqdm(perros.items(), desc='Perros'):
    H[perro] = []
    for r in rutas:
        try:
            v, ok = huella(r)
        except Exception as e:
            print('Foto omitida', r, e)
            continue
        H[perro].append(v)
        detectadas += ok
H = {k: np.stack(v) for k, v in H.items() if len(v) >= 2}
total = sum(len(v) for v in H.values())
print(f'Huellas: {total}. El detector encontró la cabeza en {detectadas / max(total, 1):.0%} de las fotos.')

## 5. Simular registros y búsquedas
Por cada perro, la mitad de sus fotos (hasta 5) hace de **registro del dueño** y la otra mitad de **escaneo de quien lo encuentra**. Como en la app, se compara la plantilla (promedio y cada captura) contra la búsqueda y vale el par más parecido. Cada búsqueda se compara con **todos** los perros registrados.

Se miden dos casos: quien encuentra hace el escaneo completo (varias capturas, como la app hoy) y quien encuentra tiene **una sola foto**.

In [ ]:
def plantilla(m):
    p = m.mean(0)
    return np.vstack([p / (np.linalg.norm(p) or 1), m])

def puntajes(registros, busquedas):
    # Matriz [registro, búsqueda] con el máximo entre todos los pares de capturas.
    A = np.vstack(registros); B = np.vstack(busquedas)
    ia = np.cumsum([0] + [len(r) for r in registros])[:-1]
    ib = np.cumsum([0] + [len(b) for b in busquedas])[:-1]
    S = np.empty((len(registros), len(busquedas)), np.float32)
    for j0 in range(0, len(busquedas), 256):
        j1 = min(j0 + 256, len(busquedas))
        cols = slice(ib[j0], ib[j1] if j1 < len(busquedas) else len(B))
        sub = A @ B[cols].T
        sub = np.maximum.reduceat(sub, ia, axis=0)
        S[:, j0:j1] = np.maximum.reduceat(sub, ib[j0:j1] - ib[j0], axis=1)
    return S

def evaluar(S):
    n = len(S)
    real = np.diag(S)
    otros = S[~np.eye(n, dtype=bool)]
    orden = np.argsort(-S, axis=0)
    mejor = orden[0]
    pos = (orden == np.arange(n)).argmax(0)  # puesto del perro correcto
    top = S[mejor, np.arange(n)]
    correcto = mejor == np.arange(n)
    auto_ok = correcto & (top >= UMBRAL)
    auto_mal = ~correcto & (top >= UMBRAL)
    # Sin aviso correcto: ¿el perro correcto sale entre las 3 sugerencias (>= 0.63)?
    # Si hubo un aviso equivocado, ese queda fuera de las sugerencias.
    sugerido = ~auto_ok & (pos < np.where(auto_mal, 4, 3)) & (real >= SUGERENCIA)
    tabla = []
    for u in np.round(np.arange(0.60, 0.951, 0.02), 2):
        tabla.append({'umbral': float(u),
                      'mismo_perro_supera': float((real >= u).mean()),
                      'perro_distinto_supera': float((otros >= u).mean()),
                      'aviso_correcto': float((correcto & (top >= u)).mean()),
                      'aviso_equivocado': float((~correcto & (top >= u)).mean())})
    return {
        'perros_comparados': int(n),
        'primer_lugar': float((pos == 0).mean()),
        'entre_los_5_primeros': float((pos < 5).mean()),
        'aviso_automatico_correcto': float(auto_ok.mean()),
        'aviso_automatico_equivocado': float(auto_mal.mean()),
        'sugerido_para_confirmar': float(sugerido.mean()),
        'no_aparece': float((~auto_ok & ~sugerido).mean()),
        'mismo_perro_puntaje_mediano': float(np.median(real)),
        'perro_distinto_puntaje_mediano': float(np.median(otros)),
        'tabla_umbrales': tabla,
        '_real': real, '_otros': otros,
    }

rng = random.Random(SEMILLA)
registros, completas, una = [], [], []
for perro, m in H.items():
    idx = list(range(len(m))); rng.shuffle(idx)
    mitad = (len(idx) + 1) // 2
    reg, bus = idx[:mitad][:5], idx[mitad:][:5]
    registros.append(plantilla(m[reg]))
    completas.append(plantilla(m[bus]))
    una.append(m[bus[:1]])

R = {'completo': evaluar(puntajes(registros, completas)),
     'una_foto': evaluar(puntajes(registros, una))}
print('Listo')

## 6. Resultados

In [ ]:
import matplotlib.pyplot as plt

pct = lambda x: f'{x * 100:.1f}%'
def umbral_para(r, maximo=0.01):
    # Umbral más bajo con avisos equivocados bajo el máximo (1% de las búsquedas).
    for f in r['tabla_umbrales']:
        if f['aviso_equivocado'] <= maximo:
            return f
for caso, nombre in [('completo', 'Escaneo completo (como la app)'), ('una_foto', 'Una sola foto de quien lo encuentra')]:
    r = R[caso]
    print(f"\n=== {nombre}: {r['perros_comparados']} perros ===")
    print('El perro correcto sale primero:          ', pct(r['primer_lugar']))
    print('El perro correcto sale entre los 5 primeros:', pct(r['entre_los_5_primeros']))
    print(f'Con el umbral de la app ({UMBRAL}):')
    print('  aviso automático al dueño correcto:     ', pct(r['aviso_automatico_correcto']))
    print('  sugerido para que una persona confirme: ', pct(r['sugerido_para_confirmar']))
    print('  el perro correcto no aparece:           ', pct(r['no_aparece']))
    print('  además, aviso a un dueño equivocado:    ', pct(r['aviso_automatico_equivocado']))
    f = umbral_para(r)
    if f:
        print(f"Umbral sugerido (máximo 1% de avisos equivocados): {f['umbral']:.2f} -> aviso correcto {pct(f['aviso_correcto'])}")

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
for ax, caso, titulo in [(ejes[0], 'completo', 'Escaneo completo'), (ejes[1], 'una_foto', 'Una sola foto')]:
    r = R[caso]
    ax.hist(r['_otros'], bins=60, range=(0, 1), density=True, alpha=0.6, color='#999', label='Perros distintos')
    ax.hist(r['_real'], bins=60, range=(0, 1), density=True, alpha=0.7, color='#e8743b', label='El mismo perro')
    for u, c in [(SUGERENCIA, '#f0b429'), (UMBRAL, '#d9573b')]:
        ax.axvline(u, color=c, ls='--')
    ax.set_title(f"{titulo}: primer lugar {pct(r['primer_lugar'])}")
    ax.set_xlabel('Parecido (similitud coseno)'); ax.legend()
plt.tight_layout(); plt.savefig('grafico_reconocimiento.png', dpi=150); plt.show()

limpio = {k: {a: b for a, b in v.items() if not a.startswith('_')} for k, v in R.items()}
limpio['datos'] = {'set': 'DogFaceNet', 'perros': len(H), 'fotos': int(sum(len(v) for v in H.values())),
                   'cabeza_detectada': detectadas / max(total, 1), 'umbral_app': UMBRAL}
json.dump(limpio, open('resultados_reconocimiento.json', 'w'), indent=2, ensure_ascii=False)

c = R['completo']
print('\nTexto para la postulación (revísalo antes de usarlo):\n')
print(f"En una prueba con {c['perros_comparados']} perros del set público DogFaceNet, comparando cada perro contra todos "
      f"los demás, el motor de Kiltrazo ubicó al perro correcto en primer lugar en el {pct(c['primer_lugar'])} de los casos "
      f"y entre los 5 primeros en el {pct(c['entre_los_5_primeros'])}, sin entrenamiento específico para perros. "
      "El ajuste fino con fotos propias es el siguiente paso.")

if 'google.colab' in sys.modules:
    from google.colab import files
    files.download('resultados_reconocimiento.json'); files.download('grafico_reconocimiento.png')